<a href="https://colab.research.google.com/github/TomekBM/MEDICA/blob/main/MEDICA%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install flask-sqlalchemy flask-login

ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information about how to avoid this problem.


ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information about how to avoid this problem.


In [12]:
# Re-installing Flask extensions to ensure they are available in the environment.
!pip install flask-sqlalchemy flask-login

### 1. Project Initialization & File Setup
This cell creates the directory structure and the CSS file for our medical theme.

In [13]:
import os

# Tworzenie struktury katalogów
os.makedirs('templates', exist_ok=True)
os.makedirs('static/css', exist_ok=True)

css_content = """
:root {
    --primary-color: #0056b3;
    --bg-color: #f4f7f6;
    --card-bg: #ffffff;
    --text-color: #333;
    --border-color: #e0e0e0;
}

body {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    background-color: var(--bg-color);
    color: var(--text-color);
    margin: 0;
    padding: 0;
}

.navbar {
    background-color: var(--primary-color);
    color: white;
    padding: 1rem 2rem;
    display: flex;
    justify-content: space-between;
    align-items: center;
    box-shadow: 0 2px 4px rgba(0,0,0,0.1);
}

.container {
    max-width: 1000px;
    margin: 2rem auto;
    padding: 0 1rem;
}

.card {
    background: var(--card-bg);
    border-radius: 8px;
    box-shadow: 0 4px 6px rgba(0,0,0,0.05);
    padding: 2rem;
    margin-bottom: 1rem;
}

.login-container {
    max-width: 400px;
    margin: 5rem auto;
}

.btn {
    background-color: var(--primary-color);
    color: white;
    border: none;
    padding: 0.75rem 1.5rem;
    border-radius: 4px;
    cursor: pointer;
    width: 100%;
    font-size: 1rem;
}

.btn:hover { background-color: #004494; }

input, select, textarea {
    width: 100%;
    padding: 0.8rem;
    margin: 0.5rem 0 1.2rem 0;
    border: 1px solid var(--border-color);
    border-radius: 4px;
    box-sizing: border-box;
}

.tabs {
    display: flex;
    border-bottom: 2px solid var(--border-color);
    margin-bottom: 1rem;
}

.tab-link {
    padding: 10px 20px;
    cursor: pointer;
    border: none;
    background: none;
    font-weight: bold;
    color: #666;
}

.tab-link.active {
    color: var(--primary-color);
    border-bottom: 3px solid var(--primary-color);
}

.tab-content {
    display: none;
}

.tab-content.active {
    display: block;
}

.data-row {
    display: flex;
    justify-content: space-between;
    padding: 0.8rem 0;
    border-bottom: 1px solid #eee;
}

.data-label { font-weight: bold; color: #555; }
"""

with open('static/css/style.css', 'w') as f:
    f.write(css_content)
print('Plik CSS został wygenerowany.')

Plik CSS został wygenerowany.


### 2. HTML Templates (Polish UI)
We define the base layout, registration, login, and the dashboard with tabs.

In [14]:
templates = {
    'base.html': """
<!DOCTYPE html>
<html lang="pl">
<head>
    <meta charset="UTF-8">
    <title>Medica+ - {% block title %}{% endblock %}</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
</head>
<body>
    <nav class="navbar">
        <div class="logo"><strong>Medica+</strong> | System Kliniki</div>
        <div>
            {% if current_user.is_authenticated %}
                <span>Witaj, {{ current_user.name }}</span> |
                <a href="{{ url_for('logout') }}" style="color:white">Wyloguj</a>
            {% else %}
                <a href="{{ url_for('login') }}" style="color:white">Logowanie</a> |
                <a href="{{ url_for('register') }}" style="color:white">Rejestracja</a>
            {% endif %}
        </div>
    </nav>
    <div class="container">
        {% with messages = get_flashed_messages() %}
          {% if messages %}
            {% for message in messages %}<div class="card" style="color:red">{{ message }}</div>{% endfor %}
          {% endif %}
        {% endwith %}
        {% block content %}{% endblock %}
    </div>
</body>
</html>
""",
    'login.html': """
{% extends 'base.html' %}
{% block title %}Logowanie{% endblock %}
{% block content %}
<div class="card login-container">
    <h2 style="text-align:center">Logowanie do systemu</h2>
    <form method="POST">
        <label>Adres e-mail</label>
        <input type="email" name="email" required placeholder="np. pacjent@poczta.pl">
        <label>Hasło</label>
        <input type="password" name="password" required>
        <button type="submit" class="btn">Zaloguj się</button>
    </form>
    <p style="text-align:center">Nie masz konta? <a href="{{ url_for('register') }}">Zarejestruj się</a></p>
</div>
{% endblock %}
""",
    'register.html': """
{% extends 'base.html' %}
{% block title %}Nowy Pacjent{% endblock %}
{% block content %}
<div class="card">
    <h2>Dodaj nowego pacjenta</h2>
    <form method="POST">
        <div style="display:grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div>
                <label>Imię i nazwisko</label>
                <input type="text" name="name" required>
                <label>PESEL</label>
                <input type="text" name="pesel" required maxlength="11">
                <label>Data urodzenia</label>
                <input type="date" name="birth_date" required>
                <label>Płeć</label>
                <select name="gender">
                    <option value="Kobieta">Kobieta</option>
                    <option value="Mężczyzna">Mężczyzna</option>
                </select>
            </div>
            <div>
                <label>Adres e-mail</label>
                <input type="email" name="email" required>
                <label>Numer telefonu</label>
                <input type="text" name="phone">
                <label>Adres zamieszkania</label>
                <input type="text" name="address">
                <label>Hasło dostępu</label>
                <input type="password" name="password" required>
            </div>
        </div>
        <button type="submit" class="btn">Zapisz i utwórz konto</button>
    </form>
</div>
{% endblock %}
""",
    'dashboard.html': """
{% extends 'base.html' %}
{% block title %}Karta Pacjenta{% endblock %}
{% block content %}
<div class="card">
    <h1>Karta pacjenta: {{ current_user.name }}</h1>

    <div class="tabs">
        <button class="tab-link active" onclick="openTab(event, 'dane')">Dane pacjenta</button>
        <button class="tab-link" onclick="openTab(event, 'dokumentacja')">Dokumentacja medyczna</button>
        <button class="tab-link" onclick="openTab(event, 'wizyty')">Wizyty</button>
        <button class="tab-link" onclick="openTab(event, 'recepty')">Recepty</button>
    </div>

    <div id="dane" class="tab-content active">
        <h3>Informacje podstawowe</h3>
        <div class="data-row"><span class="data-label">PESEL:</span> <span>{{ current_user.pesel }}</span></div>
        <div class="data-row"><span class="data-label">Data urodzenia:</span> <span>{{ current_user.birth_date }}</span></div>
        <div class="data-row"><span class="data-label">Płeć:</span> <span>{{ current_user.gender }}</span></div>
        <div class="data-row"><span class="data-label">E-mail:</span> <span>{{ current_user.email }}</span></div>
        <div class="data-row"><span class="data-label">Telefon:</span> <span>{{ current_user.phone }}</span></div>
        <div class="data-row"><span class="data-label">Adres:</span> <span>{{ current_user.address }}</span></div>
    </div>

    <div id="dokumentacja" class="tab-content">
        <h3>Historia medyczna</h3>
        {% for record in records %}
        <div style="border: 1px solid #eee; padding: 10px; margin-bottom: 10px;">
            <p><strong>Lekarz prowadzący:</strong> {{ record.doctor }}</p>
            <p><strong>Rozpoznanie:</strong> {{ record.diagnosis }}</p>
            <p><strong>Leki:</strong> {{ record.meds }}</p>
            <p><strong>Alergie:</strong> {{ record.allergies }}</p>
        </div>
        {% endfor %}
    </div>

    <div id="wizyty" class="tab-content">
        <h3>Zaplanowane i archiwalne wizyty</h3>
        <table style="width:100%; text-align:left;">
            <thead><tr><th>Data</th><th>Gabinet</th><th>Status</th></tr></thead>
            <tbody>
                {% for v in visits %}
                <tr><td>{{ v.date }}</td><td>{{ v.room }}</td><td>{{ v.status }}</td></tr>
                {% endfor %}
            </tbody>
        </table>
    </div>

    <div id="recepty" class="tab-content">
        <h3>Aktywne recepty</h3>
        {% for r in prescriptions %}
        <div style="background: #f9f9f9; padding: 10px; border-left: 5px solid var(--primary-color); margin-bottom: 10px;">
            <p><strong>Kod:</strong> {{ r.code }} | <strong>Lek:</strong> {{ r.drug_name }}</p>
            <p><small>Wystawiono: {{ r.issue_date }}</small></p>
        </div>
        {% endfor %}
    </div>
</div>

<script>
function openTab(evt, tabName) {
    var i, tabcontent, tablinks;
    tabcontent = document.getElementsByClassName("tab-content");
    for (i = 0; i < tabcontent.length; i++) { tabcontent[i].style.display = "none"; }
    tablinks = document.getElementsByClassName("tab-link");
    for (i = 0; i < tablinks.length; i++) { tablinks[i].className = tablinks[i].className.replace(" active", ""); }
    document.getElementById(tabName).style.display = "block";
    evt.currentTarget.className += " active";
}
</script>
{% endblock %}
"""
}

for name, content in templates.items():
    with open(f'templates/{name}', 'w') as f:
        f.write(content)
print('Pliki HTML zostały wygenerowane w folderze templates/.')

Pliki HTML zostały wygenerowane w folderze templates/.


### 3. Flask Backend & SQLite Database Logic
This contains the core logic: database models, authentication, and view routing.

In [15]:
from flask import Flask, render_template, request, redirect, url_for, flash
from flask_sqlalchemy import SQLAlchemy
from flask_login import LoginManager, UserMixin, login_user, login_required, logout_user, current_user
from werkzeug.security import generate_password_hash, check_password_hash

app = Flask(__name__)
app.config['SECRET_KEY'] = 'medica-secret-key-123'
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///medica.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

db = SQLAlchemy(app)
login_manager = LoginManager(app)
login_manager.login_view = 'login'

# Database Models
class Patient(UserMixin, db.Model):
    id = db.Column(db.Integer, primary_key=True)
    email = db.Column(db.String(100), unique=True, nullable=False)
    password = db.Column(db.String(100), nullable=False)
    name = db.Column(db.String(100), nullable=False)
    pesel = db.Column(db.String(11), unique=True, nullable=False)
    birth_date = db.Column(db.String(20))
    phone = db.Column(db.String(20))
    address = db.Column(db.String(200))
    gender = db.Column(db.String(20))

class MedicalRecord(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    patient_id = db.Column(db.Integer, db.ForeignKey('patient.id'))
    doctor = db.Column(db.String(100))
    diagnosis = db.Column(db.Text)
    meds = db.Column(db.Text)
    allergies = db.Column(db.String(200))

class Visit(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    patient_id = db.Column(db.Integer, db.ForeignKey('patient.id'))
    date = db.Column(db.String(50))
    room = db.Column(db.String(10))
    status = db.Column(db.String(50))

class Prescription(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    patient_id = db.Column(db.Integer, db.ForeignKey('patient.id'))
    drug_name = db.Column(db.String(100))
    code = db.Column(db.String(10))
    issue_date = db.Column(db.String(20))

@login_manager.user_loader
def load_user(user_id):
    return Patient.query.get(int(user_id))

# Routes
@app.route('/')
def index():
    return redirect(url_for('login'))

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        email = request.form.get('email')
        password = request.form.get('password')
        user = Patient.query.filter_by(email=email).first()
        if user and check_password_hash(user.password, password):
            login_user(user)
            return redirect(url_for('dashboard'))
        flash('Błędny e-mail lub hasło.')
    return render_template('login.html')

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        email = request.form.get('email')
        pesel = request.form.get('pesel')

        if Patient.query.filter((Patient.email == email) | (Patient.pesel == pesel)).first():
            flash('Pacjent z tym adresem e-mail lub PESEL już istnieje.')
            return redirect(url_for('register'))

        new_patient = Patient(
            email=email,
            password=generate_password_hash(request.form.get('password'), method='pbkdf2:sha256'),
            name=request.form.get('name'),
            pesel=pesel,
            birth_date=request.form.get('birth_date'),
            phone=request.form.get('phone'),
            address=request.form.get('address'),
            gender=request.form.get('gender')
        )
        db.session.add(new_patient)
        db.session.commit()

        # Seed dummy data for new patient
        db.session.add(MedicalRecord(patient_id=new_patient.id, doctor="dr Jan Kowalski", diagnosis="Nadciśnienie tętnicze", meds="Amlodypina 5mg", allergies="Brak"))
        db.session.add(Visit(patient_id=new_patient.id, date="2023-12-15 10:30", room="204", status="Oczekująca"))
        db.session.add(Prescription(patient_id=new_patient.id, drug_name="Paracetamol", code="4432", issue_date="2023-11-01"))
        db.session.commit()

        flash('Konto utworzone pomyślnie! Możesz się zalogować.')
        return redirect(url_for('login'))
    return render_template('register.html')

@app.route('/dashboard')
@login_required
def dashboard():
    records = MedicalRecord.query.filter_by(patient_id=current_user.id).all()
    visits = Visit.query.filter_by(patient_id=current_user.id).all()
    prescriptions = Prescription.query.filter_by(patient_id=current_user.id).all()
    return render_template('dashboard.html', records=records, visits=visits, prescriptions=prescriptions)

@app.route('/logout')
@login_required
def logout():
    logout_user()
    return redirect(url_for('login'))

# Create DB and a test user
with app.app_context():
    db.create_all()
    if not Patient.query.filter_by(email='test@medica.pl').first():
        test_p = Patient(email='test@medica.pl', password=generate_password_hash('pacjent123'), name='Adam Testowy', pesel='90010112345', birth_date='1990-01-01', phone='555-666-777', address='ul. Polna 1, Warszawa', gender='Mężczyzna')
        db.session.add(test_p)
        db.session.commit()
        db.session.add(MedicalRecord(patient_id=test_p.id, doctor="dr Anna Nowak", diagnosis="Kontrola ogólna", meds="Witaminy", allergies="Pyłki"))
        db.session.add(Visit(patient_id=test_p.id, date="2024-01-20 08:00", room="101", status="Zaplanowana"))
        db.session.commit()


### 4. Run the Application
In Google Colab, we use `google.colab.output.serve_kernel_port_as_window` to create a public URL for our local Flask app.

In [ ]:
from google.colab.output import serve_kernel_port_as_iframe

# Set the port
PORT = 5000

# Display the link to open the app
print("Klinika Medica+ jest gotowa!")
serve_kernel_port_as_iframe(PORT)

# Start the Flask app
if __name__ == '__main__':
    app.run(port=PORT)

Klinika Medica+ jest gotowa!


<IPython.core.display.Javascript object>

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [05/May/2026 18:51:47] "GET / HTTP/1.1" 302 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 18:51:47] "GET /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 18:51:47] "GET /static/css/style.css HTTP/1.1" 200 -


In [10]:
import os
from flask import Flask, render_template, request, redirect, url_for, flash
from flask_sqlalchemy import SQLAlchemy
from flask_login import LoginManager, UserMixin, login_user, login_required, logout_user, current_user
from werkzeug.security import generate_password_hash, check_password_hash
from google.colab.output import serve_kernel_port_as_iframe

# 1. Tworzenie struktury katalogów i plików CSS
# (W razie potrzeby, upewnij się, że katalogi i pliki są na miejscu)
os.makedirs('templates', exist_ok=True)
os.makedirs('static/css', exist_ok=True)

css_content = """
:root {
    --primary-color: #0056b3;
    --bg-color: #f4f7f6;
    --card-bg: #ffffff;
    --text-color: #333;
    --border-color: #e0e0e0;
}

body {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    background-color: var(--bg-color);
    color: var(--text-color);
    margin: 0;
    padding: 0;
}

.navbar {
    background-color: var(--primary-color);
    color: white;
    padding: 1rem 2rem;
    display: flex;
    justify-content: space-between;
    align-items: center;
    box-shadow: 0 2px 4px rgba(0,0,0,0.1);
}

.container {
    max-width: 1000px;
    margin: 2rem auto;
    padding: 0 1rem;
}

.card {
    background: var(--card-bg);
    border-radius: 8px;
    box-shadow: 0 4px 6px rgba(0,0,0,0.05);
    padding: 2rem;
    margin-bottom: 1rem;
}

.login-container {
    max-width: 400px;
    margin: 5rem auto;
}

.btn {
    background-color: var(--primary-color);
    color: white;
    border: none;
    padding: 0.75rem 1.5rem;
    border-radius: 4px;
    cursor: pointer;
    width: 100%;
    font-size: 1rem;
}

.btn:hover { background-color: #004494; }

input, select, textarea {
    width: 100%;
    padding: 0.8rem;
    margin: 0.5rem 0 1.2rem 0;
    border: 1px solid var(--border-color);
    border-radius: 4px;
    box-sizing: border-box;
}

.tabs {
    display: flex;
    border-bottom: 2px solid var(--border-color);
    margin-bottom: 1rem;
}

.tab-link {
    padding: 10px 20px;
    cursor: pointer;
    border: none;
    background: none;
    font-weight: bold;
    color: #666;
}

.tab-link.active {
    color: var(--primary-color);
    border-bottom: 3px solid var(--primary-color);
}

.tab-content {
    display: none;
}

.tab-content.active {
    display: block;
}

.data-row {
    display: flex;
    justify-content: space-between;
    padding: 0.8rem 0;
    border-bottom: 1px solid #eee;
}

.data-label { font-weight: bold; color: #555; }
"""

with open('static/css/style.css', 'w') as f:
    f.write(css_content)
print('Plik CSS został wygenerowany.')

# 2. Tworzenie plików HTML z szablonami
templates = {
    'base.html': """
<!DOCTYPE html>
<html lang=\"pl\">
<head>
    <meta charset=\"UTF-8\">
    <title>Medica+ - {% block title %}{% endblock %}</title>
    <link rel=\"stylesheet\" href=\"{{ url_for('static', filename='css/style.css') }}\">
</head>
<body>
    <nav class=\"navbar\">
        <div class=\"logo\"><strong>Medica+</strong> | System Kliniki</div>
        <div>
            {% if current_user.is_authenticated %}
                <span>Witaj, {{ current_user.name }}</span> |
                <a href=\"{{ url_for('logout') }}\" style=\"color:white\">Wyloguj</a>
            {% else %}
                <a href=\"{{ url_for('login') }}\" style=\"color:white\">Logowanie</a> |
                <a href=\"{{ url_for('register') }}\" style=\"color:white\">Rejestracja</a>
            {% endif %}
        </div>
    </nav>
    <div class=\"container\">
        {% with messages = get_flashed_messages() %}
          {% if messages %}
            {% for message in messages %}<div class=\"card\" style=\"color:red\">{{ message }}</div>{% endfor %}
          {% endif %}
        {% endwith %}
        {% block content %}{% endblock %}
    </div>
</body>
</html>
""",
    'login.html': """
{% extends 'base.html' %}
{% block title %}Logowanie{% endblock %}
{% block content %}
<div class=\"card login-container\">
    <h2 style=\"text-align:center\">Logowanie do systemu</h2>
    <form method=\"POST\">
        <label>Adres e-mail</label>
        <input type=\"email\" name=\"email\" required placeholder=\"np. pacjent@poczta.pl\">
        <label>Hasło</label>
        <input type=\"password\" name=\"password\" required>
        <button type=\"submit\" class=\"btn\">Zaloguj się</button>
    </form>
    <p style=\"text-align:center\">Nie masz konta? <a href=\"{{ url_for('register') }}\">Zarejestruj się</a></p>
</div>
{% endblock %}
""",
    'register.html': """
{% extends 'base.html' %}
{% block title %}Nowy Pacjent{% endblock %}
{% block content %}
<div class=\"card\">
    <h2>Dodaj nowego pacjenta</h2>
    <form method=\"POST\">
        <div style=\"display:grid; grid-template-columns: 1fr 1fr; gap: 20px;\">
            <div>
                <label>Imię i nazwisko</label>
                <input type=\"text\" name=\"name\" required>
                <label>PESEL</label>
                <input type=\"text\" name=\"pesel\" required maxlength=\"11\">
                <label>Data urodzenia</label>
                <input type=\"date\" name=\"birth_date\" required>
                <label>Płeć</label>
                <select name=\"gender\">
                    <option value=\"Kobieta\">Kobieta</option>
                    <option value=\"Mężczyzna\">Mężczyzna</option>
                </select>
            </div>
            <div>
                <label>Adres e-mail</label>
                <input type=\"email\" name=\"email\" required>
                <label>Numer telefonu</label>
                <input type=\"text\" name=\"phone\">
                <label>Adres zamieszkania</label>
                <input type=\"text\" name=\"address\">
                <label>Hasło dostępu</label>
                <input type=\"password\" name=\"password\" required>
            </div>
        </div>
        <button type=\"submit\" class=\"btn\">Zapisz i utwórz konto</button>
    </form>
</div>
{% endblock %}
""",
    'dashboard.html': """
{% extends 'base.html' %}
{% block title %}Karta Pacjenta{% endblock %}
{% block content %}
<div class=\"card\">
    <h1>Karta pacjenta: {{ current_user.name }}</h1>

    <div class=\"tabs\">
        <button class=\"tab-link active\" onclick=\"openTab(event, 'dane')\">Dane pacjenta</button>
        <button class=\"tab-link\" onclick=\"openTab(event, 'dokumentacja')\">Dokumentacja medyczna</button>
        <button class=\"tab-link\" onclick=\"openTab(event, 'wizyty')\">Wizyty</button>
        <button class=\"tab-link\" onclick=\"openTab(event, 'recepty')\">Recepty</button>
    </div>

    <div id=\"dane\" class=\"tab-content active\">
        <h3>Informacje podstawowe</h3>
        <div class=\"data-row\"><span class=\"data-label\">PESEL:</span> <span>{{ current_user.pesel }}</span></div>
        <div class=\"data-row\"><span class=\"data-label\">Data urodzenia:</span> <span>{{ current_user.birth_date }}</span></div>
        <div class=\"data-row\"><span class=\"data-label\">Płeć:</span> <span>{{ current_user.gender }}</span></div>
        <div class=\"data-row\"><span class=\"data-label\">E-mail:</span> <span>{{ current_user.email }}</span></div>
        <div class=\"data-row\"><span class=\"data-label\">Telefon:</span> <span>{{ current_user.phone }}</span></div>
        <div class=\"data-row\"><span class=\"data-label\">Adres:</span> <span>{{ current_user.address }}</span></div>
    </div>

    <div id=\"dokumentacja\" class=\"tab-content\">
        <h3>Historia medyczna</h3>
        {% for record in records %}
        <div style=\"border: 1px solid #eee; padding: 10px; margin-bottom: 10px;\">
            <p><strong>Lekarz prowadzący:</strong> {{ record.doctor }}</p>
            <p><strong>Rozpoznanie:</strong> {{ record.diagnosis }}</p>
            <p><strong>Leki:</strong> {{ record.meds }}</p>
            <p><strong>Alergie:</strong> {{ record.allergies }}</p>
        </div>
        {% endfor %}
    </div>

    <div id=\"wizyty\" class=\"tab-content\">
        <h3>Zaplanowane i archiwalne wizyty</h3>
        <table style=\"width:100%; text-align:left;\">
            <thead><tr><th>Data</th><th>Gabinet</th><th>Status</th></tr></thead>
            <tbody>
                {% for v in visits %}
                <tr><td>{{ v.date }}</td><td>{{ v.room }}</td><td>{{ v.status }}</td></tr>
                {% endfor %}
            </tbody>
        </table>
    </div>

    <div id=\"recepty\" class=\"tab-content\">
        <h3>Aktywne recepty</h3>
        {% for r in prescriptions %}
        <div style=\"background: #f9f9f9; padding: 10px; border-left: 5px solid var(--primary-color); margin-bottom: 10px;\">
            <p><strong>Kod:</strong> {{ r.code }} | <strong>Lek:</strong> {{ r.drug_name }}</p>
            <p><small>Wystawiono: {{ r.issue_date }}</small></p>
        </div>
        {% endfor %}
    </div>
</div>

<script>
function openTab(evt, tabName) {
    var i, tabcontent, tablinks;
    tabcontent = document.getElementsByClassName(\"tab-content\");
    for (i = 0; i < tabcontent.length; i++) { tabcontent[i].style.display = \"none\"; }
    tablinks = document.getElementsByClassName(\"tab-link\");
    for (i = 0; i < tablinks.length; i++) { tablinks[i].className = tablinks[i].className.replace(\" active\", \"\"); }
    document.getElementById(tabName).style.display = \"block\";
    evt.currentTarget.className += \" active\";
}
</script>
{% endblock %}
"""
}

for name, content in templates.items():
    with open(f'templates/{name}', 'w') as f:
        f.write(content)
print('Pliki HTML zostały wygenerowane w folderze templates/.')

# 3. Logika backendu Flask i baza danych SQLite
# Usuń istniejące instancje aplikacji, aby zapobiec konfliktom
if 'app' in globals():
    del app
if 'db' in globals():
    del db
if 'login_manager' in globals():
    del login_manager

app = Flask(__name__)
app.config['SECRET_KEY'] = 'medica-secret-key-123'
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///medica.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

db = SQLAlchemy(app)
login_manager = LoginManager(app)
login_manager.login_view = 'login'

# Modele bazy danych
class Patient(UserMixin, db.Model):
    id = db.Column(db.Integer, primary_key=True)
    email = db.Column(db.String(100), unique=True, nullable=False)
    password = db.Column(db.String(100), nullable=False)
    name = db.Column(db.String(100), nullable=False)
    pesel = db.Column(db.String(11), unique=True, nullable=False)
    birth_date = db.Column(db.String(20))
    phone = db.Column(db.String(20))
    address = db.Column(db.String(200))
    gender = db.Column(db.String(20))

class MedicalRecord(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    patient_id = db.Column(db.Integer, db.ForeignKey('patient.id'))
    doctor = db.Column(db.String(100))
    diagnosis = db.Column(db.Text)
    meds = db.Column(db.Text)
    allergies = db.Column(db.String(200))

class Visit(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    patient_id = db.Column(db.Integer, db.ForeignKey('patient.id'))
    date = db.Column(db.String(50))
    room = db.Column(db.String(10))
    status = db.Column(db.String(50))

class Prescription(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    patient_id = db.Column(db.Integer, db.ForeignKey('patient.id'))
    drug_name = db.Column(db.String(100))
    code = db.Column(db.String(10))
    issue_date = db.Column(db.String(20))

@login_manager.user_loader
def load_user(user_id):
    return Patient.query.get(int(user_id))

# Trasy (Routes)
@app.route('/')
def index():
    return redirect(url_for('login'))

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        email = request.form.get('email')
        password = request.form.get('password')
        user = Patient.query.filter_by(email=email).first()
        if user and check_password_hash(user.password, password):
            login_user(user)
            return redirect(url_for('dashboard'))
        flash('Błędny e-mail lub hasło.')
    return render_template('login.html')

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        email = request.form.get('email')
        pesel = request.form.get('pesel')

        if Patient.query.filter((Patient.email == email) | (Patient.pesel == pesel)).first():
            flash('Pacjent z tym adresem e-mail lub PESEL już istnieje.')
            return redirect(url_for('register'))

        new_patient = Patient(
            email=email,
            password=generate_password_hash(request.form.get('password'), method='pbkdf2:sha256'),
            name=request.form.get('name'),
            pesel=pesel,
            birth_date=request.form.get('birth_date'),
            phone=request.form.get('phone'),
            address=request.form.get('address'),
            gender=request.form.get('gender')
        )
        db.session.add(new_patient)
        db.session.commit()

        # Dodaj przykładowe dane dla nowego pacjenta
        db.session.add(MedicalRecord(patient_id=new_patient.id, doctor="dr Jan Kowalski", diagnosis="Nadciśnienie tętnicze", meds="Amlodypina 5mg", allergies="Brak"))
        db.session.add(Visit(patient_id=new_patient.id, date="2023-12-15 10:30", room="204", status="Oczekująca"))
        db.session.add(Prescription(patient_id=new_patient.id, drug_name="Paracetamol", code="4432", issue_date="2023-11-01"))
        db.session.commit()

        flash('Konto utworzone pomyślnie! Możesz się zalogować.')
        return redirect(url_for('login'))
    return render_template('register.html')

@app.route('/dashboard')
@login_required
def dashboard():
    records = MedicalRecord.query.filter_by(patient_id=current_user.id).all()
    visits = Visit.query.filter_by(patient_id=current_user.id).all()
    prescriptions = Prescription.query.filter_by(patient_id=current_user.id).all()
    return render_template('dashboard.html', records=records, visits=visits, prescriptions=prescriptions)

@app.route('/logout')
@login_required
def logout():
    logout_user()
    return redirect(url_for('login'))

# Utwórz bazę danych i testowego użytkownika
with app.app_context():
    db.create_all()
    if not Patient.query.filter_by(email='test@medica.pl').first():
        test_p = Patient(email='test@medica.pl', password=generate_password_hash('pacjent123'), name='Adam Testowy', pesel='90010112345', birth_date='1990-01-01', phone='555-666-777', address='ul. Polna 1, Warszawa', gender='Mężczyzna')
        db.session.add(test_p)
        db.session.commit()
        db.session.add(MedicalRecord(patient_id=test_p.id, doctor="dr Anna Nowak", diagnosis="Kontrola ogólna", meds="Witaminy", allergies="Pyłki"))
        db.session.add(Visit(patient_id=test_p.id, date="2024-01-20 08:00", room="101", status="Zaplanowana"))
        db.session.commit()

# 4. Uruchomienie aplikacji
PORT = 5000

# Wyświetl link do otwarcia aplikacji
print("Klinika Medica+ jest gotowa!")
serve_kernel_port_as_iframe(PORT)

# Uruchom aplikację Flask
if __name__ == '__main__':
    app.run(port=PORT)

Plik CSS został wygenerowany.
Pliki HTML zostały wygenerowane w folderze templates/.
Klinika Medica+ jest gotowa!


<IPython.core.display.Javascript object>

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [05/May/2026 18:40:13] "GET / HTTP/1.1" 302 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 18:40:14] "GET /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/May/2026 18:40:14] "GET /static/css/style.css HTTP/1.1" 200 -


In [ ]:
try:
    if 'app' in locals() or 'app' in globals():
        print(f"'app' variable is initialized: {type(app)}")
    else:
        print("'app' variable is NOT initialized.")
except NameError:
    print("'app' variable is NOT initialized (NameError).")